## Rule 3: Hiding Path Information

Based on the patterns in the BIRD dev set, we will identify tables that serve as bridges and exclude their information from the corresponding questions in the BIRD dev set

### Important Note

**_To generate the queries, you must include one development database ID in the first cell at a time. Run all the cells except for the last one each time._**

## 1. Retrieving the dev patterns for each database
The following cells retrieve the join patterns extracted from the development set queries of the BIRD benchmark. For this rule, we have divided these patterns into two sets: a training set and a test set. For now, we are working with the training patterns, which are stored in the `rule_original_patterns/train` folder. The folder can be accessed here: [Train Patterns](https://drive.google.com/drive/folders/1i87JHEK3j348RciQNZem9OpGk2BmPuAj).

The folder contains data for each development database, including a pickle file that contains 80% of the patterns for each database (the patterns are shuffled).

In [85]:
import pickle
from pathlib import Path
import numpy as np

def retrieve_all_dev_patterns(folder_path):
    """
    Retrieve all subgraphs from the pickle files in the folder.
    
    Args:
        folder_path (str): The path to the folder containing the .pkl files.
        
    Returns:
        list: A list of all subgraphs from the folder.
    """
    folder = Path(folder_path)
    all_subgraphs = []
    for file_name in folder.glob('join_patterns_*.pkl'):
        try:
            with file_name.open('rb') as f:
                join_patterns = pickle.load(f)
                subgraphs = [subgraph for subgraph in join_patterns]
                all_subgraphs.extend(subgraphs)
        except (ValueError, KeyError, pickle.UnpicklingError) as e:
            print(f"Skipping file {file_name.name}: {e}")
    return all_subgraphs


db_ids = [
        "california_schools",
        "card_games",
        "codebase_community",
        "debit_card_specializing",
        "european_football_2",
        "financial",
        "formula_1",
        "student_club",
        "superhero",
        "thrombosis_prediction",
        "toxicology",
]
# Example usage:
database_subgraphs = {}
for db_id in db_ids:
    folder_path = f'../../../../data/rule_inputs/rules_1_2/{db_id}/'
    database_subgraphs[db_id] = retrieve_all_dev_patterns(folder_path)
    print(f"Found {len(database_subgraphs[db_id])} subgraphs for {db_id}.")

# Splitting data into train, dev, and test sets (60-20-20)
train_subgraphs = {db_id: [] for db_id in db_ids}
dev_subgraphs = {db_id: [] for db_id in db_ids}
test_subgraphs = {db_id: [] for db_id in db_ids}
for db_id in db_ids:
    train_queries = []
    dev_queries = []
    test_queries = []
    for subgraph in database_subgraphs[db_id]:
    # We need to shuffle the questions first inside equivalent_queries in order to split them randomly
        equivalent_queries = subgraph['equivalent_queries']
        np.random.shuffle(equivalent_queries)
        train_queries = equivalent_queries[:int(0.6*len(equivalent_queries))]
        dev_queries = equivalent_queries[int(0.6*len(equivalent_queries)):int(0.8*len(equivalent_queries))]
        test_queries = equivalent_queries[int(0.8*len(equivalent_queries)):]
        subgraph['equivalent_queries'] = train_queries
        train_subgraphs[db_id].append(subgraph)
        subgraph['equivalent_queries'] = dev_queries
        dev_subgraphs[db_id].append(subgraph)
        subgraph['equivalent_queries'] = test_queries
        test_subgraphs[db_id].append(subgraph)

    print(f"Train: {len(train_subgraphs[db_id])}, Dev: {len(dev_subgraphs[db_id])}, Test: {len(test_subgraphs[db_id])} for {db_id}.")
print(train_subgraphs['california_schools'])

Found 6 subgraphs for california_schools.
Found 17 subgraphs for card_games.
Found 31 subgraphs for codebase_community.
Found 15 subgraphs for debit_card_specializing.
Found 22 subgraphs for european_football_2.
Found 31 subgraphs for financial.
Found 34 subgraphs for formula_1.
Found 26 subgraphs for student_club.
Found 39 subgraphs for superhero.
Found 9 subgraphs for thrombosis_prediction.
Found 14 subgraphs for toxicology.
Train: 6, Dev: 6, Test: 6 for california_schools.
Train: 17, Dev: 17, Test: 17 for card_games.
Train: 31, Dev: 31, Test: 31 for codebase_community.
Train: 15, Dev: 15, Test: 15 for debit_card_specializing.
Train: 22, Dev: 22, Test: 22 for european_football_2.
Train: 31, Dev: 31, Test: 31 for financial.
Train: 34, Dev: 34, Test: 34 for formula_1.
Train: 26, Dev: 26, Test: 26 for student_club.
Train: 39, Dev: 39, Test: 39 for superhero.
Train: 9, Dev: 9, Test: 9 for thrombosis_prediction.
Train: 14, Dev: 14, Test: 14 for toxicology.
[{'pattern_signature': (frozense

In this cell, the schema graph for each database is retrieved from the corresponding pickle file stored in the `main_graphs_pickles` folder under `dev_dbs_unnested`. The folder can be accessed here: [Schema Graphs](https://drive.google.com/drive/folders/18a1XT3zZFJO-k28OQT7Cqcm9VlZ7qljS).


## 2. Identifying the candidate table to hide (the most central table to the query)

In [86]:
import re
import networkx as nx

def find_central_table(subgraph, schema):
    if subgraph.number_of_nodes() < 3:
        return ""
    else:
        # Compute betweenness centrality for the current subgraph
        betweenness_centrality = nx.betweenness_centrality(subgraph)
        max_betweenness = max(betweenness_centrality.values())

        # Find nodes with the maximum betweenness centrality
        candidates = [(node, score) for node, score in betweenness_centrality.items() if score == max_betweenness]

        # Select the most central candidate table
        if len(candidates) == 1:
            selected_table = candidates[0][0]
        else:
            schema_centrality = nx.degree_centrality(schema)
            selected_table = max(candidates, key=lambda x: schema_centrality.get(x[0], 0))[0]
    return selected_table

train_subgraphs_with_central_tables = {}
dev_subgraphs_with_central_tables = {}
test_subgraphs_with_central_tables = {}

for db_id, train_sbs in train_subgraphs.items():
    schema_path = f'../../../../data/graph_data/bird_graphs/pickles/{db_id}_graph.pkl'
    with open(schema_path, 'rb') as f:
        schema = pickle.load(f)
    kept_subgraphs = []
    for train_subgraph in train_sbs:
        nx_subgraph = train_subgraph.get("subgraph")
        central_table = find_central_table(nx_subgraph, schema)
        if len(central_table) != 0:
            train_subgraph["central_table"] = central_table
            components = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z][a-z])|\d+[a-zA-Z]*', central_table)
            components = [comp for comp in components if comp != '_']
            components = [comp.lower() for comp in components]
            kept_queries = []
            for query in train_subgraph['equivalent_queries']:
                train_subgraph['components'] = components
                if any(comp in query['question'].lower() for comp in components):
                    kept_queries.append(query)
            train_subgraph['equivalent_queries'] = kept_queries
            kept_subgraphs.append(train_subgraph)
    train_subgraphs_with_central_tables[db_id] = kept_subgraphs

for db_id, dev_sbs in dev_subgraphs.items():
    schema_path = f'../../../../data/graph_data/bird_graphs/pickles/{db_id}_graph.pkl'
    with open(schema_path, 'rb') as f:
        schema = pickle.load(f)
    kept_subgraphs = []
    for dev_subgraph in dev_sbs:
        nx_subgraph = dev_subgraph.get("subgraph")
        central_table = find_central_table(nx_subgraph, schema)
        if len(central_table) != 0:
            dev_subgraph["central_table"] = central_table
            components = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z][a-z])|\d+[a-zA-Z]*', central_table)
            components = [comp for comp in components if comp != '_']
            components = [comp.lower() for comp in components]
            kept_queries = []
            for query in dev_subgraph['equivalent_queries']:
                dev_subgraph['components'] = components
                if any(comp in query['question'].lower() for comp in components):
                    kept_queries.append(subgraph)
            dev_subgraph['equivalent_queries'] = kept_queries
            kept_subgraphs.append(dev_subgraph)
    dev_subgraphs_with_central_tables[db_id] = kept_subgraphs

# for db_id, test_sbs in test_subgraphs.items():
#     schema_path = f'../../../../data/graph_data/bird_graphs/pickles/{db_id}_graph.pkl'
#     with open(schema_path, 'rb') as f:
#         schema = pickle.load(f)
#     kept_subgraphs = []
#     for test_subgraph in test_sbs:
#         nx_subgraph = test_subgraph.get("subgraph")
#         central_table = find_central_table(nx_subgraph, schema)
#         if len(central_table) != 0:
#             test_subgraph["central_table"] = central_table
#             components = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z][a-z])|\d+[a-zA-Z]*', central_table)
#             components = [comp for comp in components if comp != '_']
#             components = [comp.lower() for comp in components]
#             kept_queries = []
#             for query in test_subgraph['equivalent_queries']:
#                 test_subgraph['components'] = components
#                 if any(comp in query['question'].lower() for comp in components):
#                     kept_queries.append(subgraph)
#             test_subgraph['equivalent_queries'] = kept_queries
#             kept_subgraphs.append(test_subgraph)
#     test_subgraphs_with_central_tables[db_id] = kept_subgraphs




## 4. Hiding the path information
The hiding part is done using GPT-4o mini. You can use your API key stored in a `.env` file located here.

In [88]:
import openai
import os
from dotenv import load_dotenv

load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")

def synonym_replacement(db_id, question, evidence, sql_query, candidate_table, components):
    prompt = f"""
You are in a text-to-SQL context where each question corresponds to a SQL query based on a given database schema.

Your task is to conceal any direct reference to candidate table names in the question and the evidence while keeping them meaningful.  
Use the Synonym Replacement (SR) technique to replace words or phrases linked to the table names with their synonyms.  

**Important:**  
- The synonym should not refer to the candidate table itself (e.g., if the candidate table is *cards*, do not use *playing cards* as a synonym).  
- You should hide the reference to the table name but keep the rest of the question and the evidence intact.  
- If the evidence directly mentions a table name along with a column name (e.g., cards.name), replace the table name with a suitable synonym and use pronouns or descriptive phrases that imply the source. For example: 'names of all the cards in the set refers to cards.name' becomes 'names of all the pieces in the set refers to their names.'
- Be careful with the evidence, there should be no mentioning of the candidate table name in the modified evidence.
- The evidence might be an empty string, just give back an empty string for it as well

**Details:**
- The **Candidate Table** may be a single word (e.g., *users*) or a compound name (e.g., *postHistory* or *transactions_1k*).  
- You will be provided with the **Components** — a list containing either:
  - A single item (for simple table names), or
  - Multiple items (for compound names split into parts).
- If the question contains any of these components, hide the relevant part using SR.
- The synonym used should not reference the candidate table.

**Input:**
- **Database ID:** {db_id}
- **Question:** {question}
- **Evidence:** {evidence}
- **SQL Query:** {sql_query}
- **Candidate Table:** {candidate_table}
- **Components:** {components}

**Output:**  
Return only the modified question and evidence after applying the SR technique. Do not include anything else. Your output should be in this format:
New Question: modified question
New Evidence: modified evidence if evidence is provided, otherwise an empty string
"""

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    # Get response from the LLM
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=300,
        temperature=0.7,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
    )

    # Process LLM response
    generated_text = response.choices[0].message.content.strip()
    return generated_text


train_queries_with_synonym_replacement = {}
dev_queries_with_synonym_replacement = {}
for db_id, train_sbs_with_table in train_subgraphs_with_central_tables.items():
    queries_with_synonym_replacement = []
    print("Generating new questions for", db_id)
    for train_subgraph_with_table in train_sbs_with_table:
        for query_in_dev in train_subgraph_with_table['equivalent_queries']:
            new_rows = synonym_replacement(db_id, query_in_dev['question'], query_in_dev['evidence'], query_in_dev['SQL'], train_subgraph_with_table['central_table'], train_subgraph_with_table['components'])
            for row in new_rows.split("\n"):
                if row.startswith("New Question:"):
                    new_question = row.replace("New Question: ", "").strip()
                if row.startswith("New Evidence:"):
                    new_evidence = row.replace("New Evidence: ", "").strip()
            if not any(comp in new_question.lower() for comp in train_subgraph_with_table['components']):
                query_in_dev['new_question'] = new_question
                query_in_dev['new_evidence'] = new_evidence
                queries_with_synonym_replacement.append(query_in_dev)
    train_queries_with_synonym_replacement[db_id] = queries_with_synonym_replacement

for db_id, dev_sbs_with_table in dev_subgraphs_with_central_tables.items():
    queries_with_synonym_replacement = []
    print("Generating new questions for", db_id)
    for dev_subgraph_with_table in dev_sbs_with_table:
        for query_in_dev in dev_subgraph_with_table['equivalent_queries']:
            new_rows = synonym_replacement(db_id, query_in_dev['question'], query_in_dev['evidence'], query_in_dev['SQL'], dev_subgraph_with_table['central_table'], dev_subgraph_with_table['components'])
            for row in new_rows.split("\n"):
                if row.startswith("New Question:"):
                    new_question = row.replace("New Question: ", "").strip()
                if row.startswith("New Evidence:"):
                    new_evidence = row.replace("New Evidence: ", "").strip()
            if not any(comp in new_question.lower() for comp in dev_subgraph_with_table['components']):
                query_in_dev['new_question'] = new_question
                query_in_dev['new_evidence'] = new_evidence
                queries_with_synonym_replacement.append(query_in_dev)
    dev_queries_with_synonym_replacement[db_id] = queries_with_synonym_replacement

Generating new questions for california_schools
Generating new questions for card_games
[{'Tables': [{'table_name': 'cards', 'table_alias': 'T1'}, {'table_name': 'legalities', 'table_alias': 'T2'}, {'table_name': 'foreign_data', 'table_alias': 'T3'}], 'Join_relations_with_aliases': ['T1.uuid = T2.uuid', 'T1.uuid = T3.uuid'], 'Join_relations_without_aliases': ['cards.uuid = legalities.uuid', 'cards.uuid = foreign_data.uuid'], 'db_id': 'card_games', 'SQL': "SELECT T1.uuid, T3.language FROM cards AS T1 INNER JOIN legalities AS T2 ON T1.uuid = T2.uuid INNER JOIN foreign_data AS T3 ON T1.uuid = T3.uuid WHERE T2.format = 'legacy'", 'question': 'List down the uuid for legacy cards and the foreign language of these cards.', 'evidence': "legacy card refers to format = 'legacy'; foreign language refers to language in foreign_data", 'difficulty': 'simple', 'flattened_query': ''}]
dict_keys(['Tables', 'Join_relations_with_aliases', 'Join_relations_without_aliases', 'db_id', 'SQL', 'question', 'evi

KeyboardInterrupt: 

In [52]:
import openai
import os
from dotenv import load_dotenv

load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")

def backtranslation(db_id, question, evidence, sql_query, candidate_table, components):
    prompt = f"""
You are in a text-to-SQL context where each question corresponds to a SQL query based on a given database schema.

Your task is to conceal any direct reference to the candidate table names in the question and the evidence while keeping it meaningful.  
Use the **Back Translation (BT)** technique. First, translate the question and the evidence into French, then back to English. During the French translation, replace any reference to the candidate table with a relevant synonym.  

**Important:**  
- Do not use synonyms that refer directly to the candidate table (e.g., if the candidate table is *cards*, do not use *cartes de jeu*).  
- Hide only the table name reference and keep the rest of the question and evidence intact.  
- If the evidence directly mentions a table name along with a column name (e.g., cards.name), replace the table name with a suitable synonym and use pronouns or descriptive phrases that imply the source. For example: 'names of all the cards in the set refers to cards.name' becomes 'names of all the pieces in the set refers to their names.'
- Be careful with the evidence, there should be no mentioning of the candidate table name in the modified evidence.
- The evidence might be an empty string, just give back an empty string for it as well

**Details:**
- The **Candidate Table** may be a single word (e.g., *users*) or a compound name (e.g., *postHistory* or *transactions_1k*).  
- You will be provided with the **Components** — a list containing either:
  - A single item (for simple table names), or
  - Multiple items (for compound names split into parts).
- During the French translation, replace the relevant part with the synonym, then translate it back to English.
- The synonym used should not refer directly to the candidate table.


**Input:**
- **Database ID:** {db_id}
- **Question:** {question}
- **Evidence:** {evidence}
- **SQL Query:** {sql_query}
- **Candidate Table:** {candidate_table}
- **Components:** {components}

**Output:**  
Return the question and evidence in French and then in English. Do not include anything else. The output should be in this format:
Question in French: The question in French 
Question in English: The question in English
Evidence in French: The evidence in French if evidence is provided, otherwise an empty string
Evidence in English: The evidence in English if evidence is provided, otherwise an empty string
"""

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    # Get response from the LLM
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=300,
        temperature=0.7,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
    )

    # Process LLM response
    generated_text = response.choices[0].message.content.strip()
    return generated_text

train_queries_with_backtranslation = {}
dev_queries_with_backtranslation = {}


for db_id, train_sbs_with_table in train_subgraphs_with_central_tables.items():
    queries_with_backtranslation = []
    print("Generating new questions for", db_id)
    for train_subgraph_with_table in train_sbs_with_table:
        for query_in_dev in train_subgraph_with_table['equivalent_queries']:
            new_rows = backtranslation(db_id, query_in_dev['question'], query_in_dev['evidence'], query_in_dev['SQL'], train_subgraph_with_table['central_table'], train_subgraph_with_table['components'])
            for row in new_rows.split("\n"):
                if row.startswith("Question in English:"):
                    new_question = row.replace("Question in English: ", "").strip()
                if row.startswith("Evidence in English:"):
                    new_evidence = row.replace("Evidence in English: ", "").strip()
            if not any(comp in new_question.lower() for comp in train_subgraph_with_table['components']):
                query_in_dev['new_question'] = new_question
                query_in_dev['new_evidence'] = new_evidence
                queries_with_backtranslation.append(query_in_dev)
    train_queries_with_backtranslation[db_id] = queries_with_backtranslation

for db_id, dev_sbs_with_table in dev_subgraphs_with_central_tables.items():
    queries_with_backtranslation = []
    print("Generating new questions for", db_id)
    for dev_subgraph_with_table in dev_sbs_with_table:
        for query_in_dev in dev_subgraph_with_table['equivalent_queries']:
            new_rows = backtranslation(db_id, query_in_dev['question'], query_in_dev['evidence'], query_in_dev['SQL'], dev_subgraph_with_table['central_table'], dev_subgraph_with_table['components'])
            for row in new_rows.split("\n"):
                if row.startswith("Question in English:"):
                    new_question = row.replace("Question in English: ", "").strip()
                if row.startswith("Evidence in English:"):
                    new_evidence = row.replace("Evidence in English: ", "").strip()
            if not any(comp in new_question.lower() for comp in dev_subgraph_with_table['components']):
                query_in_dev['new_question'] = new_question
                query_in_dev['new_evidence'] = new_evidence
                queries_with_backtranslation.append(query_in_dev)
    dev_queries_with_backtranslation[db_id] = queries_with_backtranslation

Generating new questions for california_schools
Generating new questions for card_games
Generating new questions for codebase_community
Generating new questions for debit_card_specializing
Generating new questions for european_football_2
Generating new questions for financial
Generating new questions for formula_1
Generating new questions for student_club
Generating new questions for superhero
Generating new questions for thrombosis_prediction
Generating new questions for toxicology
Number of questions for california_schools: 0
Number of questions for card_games: 3
Number of questions for codebase_community: 7
Number of questions for debit_card_specializing: 3
Number of questions for european_football_2: 0
Number of questions for financial: 12
Number of questions for formula_1: 6
Number of questions for student_club: 3
Number of questions for superhero: 31
Number of questions for thrombosis_prediction: 4
Number of questions for toxicology: 3
Number of new questions for california_schoo

In [53]:
import openai
import os
from dotenv import load_dotenv

load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")

def contextual_augmentation(db_id, question, evidence, sql_query, candidate_table, components):
    prompt = f"""
You are in a text-to-SQL context where each question corresponds to a SQL query based on a given database schema.

Your task is to conceal any direct reference to candidate table names in the question and the evidence while keeping it meaningful.  
Use the Contextual Augmentation technique to replace words or phrases linked to the table names with their contextually appropriate synonyms to make sure that the paraphrasing aligns with the surrounding context.  

**Important:**  
- The synonym should be relevant to the context of the question, which can be derived from the db_id. E.g. if the db_id is *card_games*, one of the synonyms for *cards* could be *playing pieces* and not *vouchers*.
- The synonym should not refer to the candidate table itself (e.g., if the candidate table is *cards*, do not use *playing cards* as a synonym).  
- You should hide the reference to the table name but keep the rest of the question intact.  
- If the evidence directly mentions a table name along with a column name (e.g., cards.name), replace the table name with a suitable synonym and use pronouns or descriptive phrases that imply the source. For example: 'names of all the cards in the set refers to cards.name' becomes 'names of all the pieces in the set refers to their names.'
- Be careful with the evidence, there should be no mentioning of the candidate table name in the modified evidence.
- The evidence might be an empty string, just give back an empty string for it as well

**Details:**
- The **Candidate Table** may be a single word (e.g., *users*) or a compound name (e.g., *postHistory* or *transactions_1k*).  
- You will be provided with the **Components** — a list containing either:
  - A single item (for simple table names), or
  - Multiple items (for compound names split into parts).
- If the question contains any of these components, hide the relevant part using SR.
- The synonym used should not reference the candidate table.

**Input:**
- **Database ID:** {db_id}
- **Question:** {question}
- **Evidence:** {evidence}
- **SQL Query:** {sql_query}
- **Candidate Table:** {candidate_table}
- **Components:** {components}

**Output:**  
Return only the modified question after applying the contextual augmentation technique. Do not include anything else. The output should be in this format:
New Question: modified question
New Evidence: modified evidence if evidence is provided, otherwise an empty string 
"""

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    # Get response from the LLM
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=300,
        temperature=0.7,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
    )

    # Process LLM response
    generated_text = response.choices[0].message.content.strip()
    return generated_text

train_queries_with_contextual_augmentation = {}
dev_queries_with_contextual_augmentation = {}

for db_id, train_sbs_with_table in train_subgraphs_with_central_tables.items():
    queries_with_contextual_augmentation = []
    print("Generating new questions for", db_id)
    for train_subgraph_with_table in train_sbs_with_table:
        for query_in_dev in train_subgraph_with_table['equivalent_queries']:
            new_rows = contextual_augmentation(db_id, query_in_dev['question'], query_in_dev['evidence'], query_in_dev['SQL'], train_subgraph_with_table['central_table'], train_subgraph_with_table['components'])
            for row in new_rows.split("\n"):
                if row.startswith("New Question:"):
                    new_question = row.replace("New Question: ", "").strip()
                if row.startswith("New Evidence:"):
                    new_evidence = row.replace("New Evidence: ", "").strip()
            if not any(comp in new_question.lower() for comp in train_subgraph_with_table['components']):
                query_in_dev['new_question'] = new_question
                query_in_dev['new_evidence'] = new_evidence
                queries_with_contextual_augmentation.append(query_in_dev)
    train_queries_with_contextual_augmentation[db_id] = queries_with_contextual_augmentation

for db_id, dev_sbs_with_table in dev_subgraphs_with_central_tables.items():
    queries_with_contextual_augmentation = []
    print("Generating new questions for", db_id)
    for dev_subgraph_with_table in dev_sbs_with_table:
        for query_in_dev in dev_subgraph_with_table['equivalent_queries']:
            new_rows = contextual_augmentation(db_id, query_in_dev['question'], query_in_dev['evidence'], query_in_dev['SQL'], dev_subgraph_with_table['central_table'], dev_subgraph_with_table['components'])
            for row in new_rows.split("\n"):
                if row.startswith("New Question:"):
                    new_question = row.replace("New Question: ", "").strip()
                if row.startswith("New Evidence:"):
                    new_evidence = row.replace("New Evidence: ", "").strip()
            if not any(comp in new_question.lower() for comp in dev_subgraph_with_table['components']):
                query_in_dev['new_question'] = new_question
                query_in_dev['new_evidence'] = new_evidence
                queries_with_contextual_augmentation.append(query_in_dev)
    dev_queries_with_contextual_augmentation[db_id] = queries_with_contextual_augmentation


Generating new questions for california_schools
Generating new questions for card_games
Generating new questions for codebase_community
Generating new questions for debit_card_specializing
Generating new questions for european_football_2
Generating new questions for financial
Generating new questions for formula_1
Generating new questions for student_club
Generating new questions for superhero
Generating new questions for thrombosis_prediction
Generating new questions for toxicology
Number of questions for california_schools: 0
Number of questions for card_games: 3
Number of questions for codebase_community: 7
Number of questions for debit_card_specializing: 3
Number of questions for european_football_2: 0
Number of questions for financial: 12
Number of questions for formula_1: 6
Number of questions for student_club: 3
Number of questions for superhero: 31
Number of questions for thrombosis_prediction: 4
Number of questions for toxicology: 3
Number of new questions for california_schoo

## 5. Final queries 
This cell should be run after generating the new questions for each `db_id`. It gathers all the previously generated data in one place. To run this, you will need the `bird_dev.json` file, which is the development set file. It can be found here: [bird_dev.json](https://drive.google.com/drive/folders/10WxERhJh5ZWbad9fwNyR_51hWkSxFKr4).


In [54]:
import os
import json

def save_questions_to_file(dict_queries, output_base_dir, technique):
    output_dir = f'{output_base_dir}/{technique}'
    os.makedirs(output_dir, exist_ok=True)  # Ensure the output directory exists
    original_questions = []
    new_questions = []
    for db_id, queries in dict_queries.items():
        for i, query in enumerate(queries):
            original_entry = {
                "question_id": i,
                "db_id": db_id,
                "question": query['question'],
                "evidence": query['evidence'],
                "SQL": query['SQL'],
                "difficulty": query['difficulty']
            }
            new_entry = {
                "question_id": i,
                "db_id": db_id,
                "question": query['new_question'],
                "evidence": query['new_evidence'],
                "SQL": query['SQL'],
                "difficulty": query['difficulty']
            }
            original_questions.append(original_entry)
            new_questions.append(new_entry)

    with open(f'{output_dir}/original_questions.json', 'w') as f:
        json.dump(original_questions, f, indent=4)
    with open(f'{output_dir}/new_questions.json', 'w') as f:
        json.dump(new_questions, f, indent=4)
    
    print(f"Saved {len(original_questions)} original questions and {len(new_questions)} new questions for {db_id} to {output_dir}.")
    

# Save the questions to files
output_base_dir = "../../../../data/rule_outputs/rule_3/train"
os.makedirs(output_base_dir, exist_ok=True)  # Ensure the output directory exists

save_questions_to_file(dev_queries_with_synonym_replacement, output_base_dir, "synonym_replacement")
save_questions_to_file(dev_queries_with_backtranslation, output_base_dir, "backtranslation")
save_questions_to_file(dev_queries_with_contextual_augmentation, output_base_dir, "contextual_augmentation")

Saved 65 original questions and 65 new questions for toxicology to ../../../../data/rule_outputs/rule_3/dev/synonym_replacement.
Saved 53 original questions and 53 new questions for toxicology to ../../../../data/rule_outputs/rule_3/dev/backtranslation.
Saved 68 original questions and 68 new questions for toxicology to ../../../../data/rule_outputs/rule_3/dev/contextual_augmentation.
